# Does reactivity help the model, not just explain the lag-1 correlation?

`3.8` (on `modelling/lead-lag-test`) tested whether the reactivity filter explains the
project's lead-lag finding and came back null -- reactive-excluded and reactive-only
articles showed statistically indistinguishable correlation at lag -1. That answers a
mechanism question, not the question that actually matters for this project: does adding
reactivity as a feature change the tuned session-level model's *predictive* accuracy.

This notebook checks that, cheaply. It takes `3.7`'s winning config exactly as tuned --
same hyperparameters, same locked holdout, same everything -- and asks only one new
question: does adding `reactive_headline` and a body-text reactivity proxy to
`SESSION_TEXT_FEATURES` change the holdout score. No new hyperparameter search, no
pipeline rebuild. If this doesn't move the needle, that is a stronger and more direct
reason to leave reactivity out than the lag-1 null result was on its own.

**Deliberately on its own branch, not `modelling/lead-lag-test`.** Different question,
pushed separately.

### 0. Setup

Same real headline + body text as the lead-lag notebook, same fast regex scoring
(`reactive.classify_reactive_sentence()`, no spaCy/pipeline dependency), merged onto the
real `model_ready_pooled.parquet`.

In [1]:
import re
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

from stock_predictor.config import PROCESSED_DATA_DIR
from stock_predictor.market.evaluate import fit_final_model, _score
from stock_predictor.text.reactive import classify_reactive_sentence

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

df = pd.read_parquet(PROCESSED_DATA_DIR / "merged" / "model_ready_pooled.parquet")
df = df.sort_values("timestamp_utc").reset_index(drop=True)

texts = []
for ticker in ["AAPL", "AMZN", "NVDA", "TSLA"]:
    t = pd.read_parquet(PROCESSED_DATA_DIR / f"{ticker}_processed_no_raw.parquet")
    t = t[["article_id", "headline", "processed_body"]].copy()
    t["ticker"] = ticker
    texts.append(t)
texts = pd.concat(texts, ignore_index=True)
df = df.merge(texts, on=["article_id", "ticker"], how="left")
print(f"pooled: {df.shape}, missing headline: {df['headline'].isna().sum()}")

2026-08-30 21:09:29.510 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: C:\Users\kacpe\OneDrive - University of Warwick\PROJECTS\Stock Predictor\stock-predictor


pooled: (41298, 39), missing headline: 0


In [2]:
def naive_split(text):
    if not isinstance(text, str) or not text.strip():
        return []
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z("])', text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= 20]


df["reactive_headline"] = df["headline"].apply(
    lambda h: classify_reactive_sentence(h).score if isinstance(h, str) else np.nan
)

t0 = time.time()
unique_bodies = df[["article_id", "processed_body"]].drop_duplicates(subset="article_id")
body_scores = {}
for aid, body in zip(unique_bodies["article_id"], unique_bodies["processed_body"]):
    sents = naive_split(body)
    if not sents:
        body_scores[aid] = (np.nan, np.nan)
        continue
    scores = [classify_reactive_sentence(s).score for s in sents]
    is_r = [s >= 1.0 for s in scores]
    body_scores[aid] = (sum(is_r) / len(sents), float(np.max(scores)))
print(f"body scoring: {len(unique_bodies)} articles in {time.time()-t0:.0f}s")

body_df = pd.DataFrame.from_dict(
    body_scores, orient="index", columns=["reactive_share", "reactive_max"]
).rename_axis("article_id").reset_index()
df = df.merge(body_df, on="article_id", how="left")
print(df[["reactive_headline", "reactive_share", "reactive_max"]].describe().round(3).to_string())

body scoring: 35678 articles in 77s
       reactive_headline  reactive_share  reactive_max
count          41298.000       41298.000     41298.000
mean               0.091           0.038         0.723
std                0.284           0.059         0.656
min                0.000           0.000         0.000
25%                0.000           0.000         0.000
50%                0.000           0.000         0.500
75%                0.000           0.060         1.000
max                3.000           1.000         4.000


### 1. The session table -- `3.7`'s exact recipe, plus three new columns

`SESSION_TEXT_FEATURES`/`SESSION_MARKET_FEATURES`/model factory ported unchanged from
`3.7`. Three additions: `reactive_headline_mean`, `reactive_share_mean` (both averaged
over the session's articles, the same way `fus_sentiment_mean` is), and
`reactive_max_loudest` (the single most reactive sentence anywhere in the session --
reactive scores are non-negative, so "loudest" is just `max`, unlike the signed
`fus_sentiment_loudest`).

In [3]:
MARKET_FEATURES = [
    "momentum_1d", "momentum_5d", "momentum_20d", "volatility_20d", "beta_20d",
    "relative_volume_20d", "daily_range_ratio_1d", "days_to_earnings", "session", "news_volume",
]
LABEL = "label_direction"
SESSION_CATEGORIES = ["pre-market", "market-hours", "after-hours"]
df["session"] = pd.Categorical(df["session"], categories=SESSION_CATEGORIES)


def signed_max_abs(values):
    values = values.dropna()
    if values.empty:
        return np.nan
    return values.iloc[values.abs().to_numpy().argmax()]


GROUP_KEYS = ["session_open", "ticker"]
MARKET_SNAPSHOT = [c for c in MARKET_FEATURES if c != "session"]
grouped = df.groupby(GROUP_KEYS, observed=True)
latest_idx = grouped["timestamp_utc"].idxmax()
snapshot = df.loc[latest_idx, GROUP_KEYS + MARKET_SNAPSHOT].set_index(GROUP_KEYS)

sessions = grouped.agg(
    label_direction=("label_direction", "first"), abnormal_return_1d=("abnormal_return_1d", "first"),
    timestamp_utc=("timestamp_utc", "max"),
    n_articles=("article_id", "size"), n_sources=("source", "nunique"),
    n_total_sents=("n_total_sents", "sum"), n_entity_sents=("n_entity_sents", "sum"),
    n_ceo_sents=("n_ceo_sents", "sum"), n_boilerplate_sents=("n_boilerplate_sents", "sum"),
    article_length=("article_length", "sum"), entity_share=("entity_share", "mean"),
    has_ceo_mention=("has_ceo_mention", "mean"),
    fus_sentiment_mean=("fus_conf_graft_floor_mean", "mean"),
    fus_sentiment_std=("fus_conf_graft_floor_mean", "std"),
    fus_sentiment_loudest=("fus_conf_graft_floor_mean", signed_max_abs),
    fus_median=("fus_conf_graft_floor_median", "mean"), fus_lead=("fus_conf_graft_floor_lead", "mean"),
    fus_top3_pos=("fus_conf_graft_floor_top3_pos", "mean"),
    fus_top3_neg=("fus_conf_graft_floor_top3_neg", "mean"),
    fus_spread=("fus_conf_graft_floor_spread", "mean"), fus_ceo_mean=("fus_ceo_mean", "mean"),
    fus_headline=("fus_headline", "mean"), fus_maxmag=("fus_maxmag", "mean"),
    fus_trusted_mean=("fus_trusted_mean", "mean"), fus_scorer_gap=("fus_scorer_gap", "mean"),
    fus_headline_gap=("fus_headline_gap", "mean"), fus_lead_gap=("fus_lead_gap", "mean"),
    reactive_headline_mean=("reactive_headline", "mean"),
    reactive_share_mean=("reactive_share", "mean"),
    reactive_max_loudest=("reactive_max", "max"),
)
mix = df.groupby(GROUP_KEYS + ["session"], observed=True).size().unstack("session", fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0)
mix.columns = [f"published_{c}_share" for c in mix.columns]
SESSION_MIX_FEATURES = list(mix.columns)
sessions = sessions.join(snapshot).join(mix).reset_index()
sessions = sessions.sort_values(["timestamp_utc", "ticker"]).reset_index(drop=True)
sessions["fus_sentiment_std"] = sessions["fus_sentiment_std"].fillna(0.0)
sessions["reactive_share_mean"] = sessions["reactive_share_mean"].fillna(0.0)
sessions["reactive_max_loudest"] = sessions["reactive_max_loudest"].fillna(0.0)

SESSION_TEXT_FEATURES = [
    "n_articles", "n_sources", "n_total_sents", "n_entity_sents", "n_ceo_sents", "n_boilerplate_sents",
    "article_length", "entity_share", "has_ceo_mention", "fus_sentiment_mean", "fus_sentiment_std",
    "fus_sentiment_loudest", "fus_median", "fus_lead", "fus_top3_pos", "fus_top3_neg", "fus_spread",
    "fus_ceo_mean", "fus_headline", "fus_maxmag", "fus_trusted_mean", "fus_scorer_gap",
    "fus_headline_gap", "fus_lead_gap",
]
REACTIVE_SESSION_FEATURES = ["reactive_headline_mean", "reactive_share_mean", "reactive_max_loudest"]
SESSION_MARKET_FEATURES = MARKET_SNAPSHOT + SESSION_MIX_FEATURES
SESSION_MODEL_FEATURES = SESSION_TEXT_FEATURES + SESSION_MARKET_FEATURES
SESSION_MODEL_FEATURES_WITH_REACTIVE = SESSION_MODEL_FEATURES + REACTIVE_SESSION_FEATURES

print(f"sessions: {sessions.shape}")
print(f"baseline features: {len(SESSION_MODEL_FEATURES)}, with reactivity: {len(SESSION_MODEL_FEATURES_WITH_REACTIVE)}")

sessions: (924, 44)


baseline features: 36, with reactivity: 39


### 2. The locked holdout -- `3.7`'s exact split

Same cut: the last fifth of sessions chronologically. Reusing it rather than cutting a
fresh one, for the same reason given earlier in this project -- there is one year of
data and no meaningfully different "future" period to hold out instead. This is a
further read of an already-read holdout; noted, not hidden.

In [4]:
HOLDOUT_FRACTION = 0.2
unique_sessions = np.sort(sessions["session_open"].unique())
n_holdout = int(np.ceil(len(unique_sessions) * HOLDOUT_FRACTION))
holdout_sessions = set(unique_sessions[-n_holdout:])

is_holdout = sessions["session_open"].isin(holdout_sessions)
tuning_pool = sessions.loc[~is_holdout].reset_index(drop=True)
holdout_pool = sessions.loc[is_holdout].reset_index(drop=True)

print(f"tuning pool: {len(tuning_pool)} rows, holdout: {len(holdout_pool)} rows")
assert len(tuning_pool) == 736 and len(holdout_pool) == 188, "must match 3.7's split exactly"


tuning pool: 736 rows, holdout: 188 rows


### 3. `3.7`'s winning config, unchanged -- baseline vs. with reactivity added

Exact hyperparameters from `3.7` section 3.4's winning trial (4023, phase grid):
`num_leaves=31, learning_rate=0.08, min_child_samples=8, n_estimators=100, subsample=1.0,
subsample_freq=3, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=5.0, max_depth=-1,
is_unbalance=False, boosting_type='dart'`. No new search -- this checks whether the three
new columns help the config that already won, not whether some other config wins with
them included.

In [5]:
def model_factory():
    return lgb.LGBMClassifier(
        objective="binary", num_leaves=31, learning_rate=0.08, min_child_samples=8,
        n_estimators=100, subsample=1.0, subsample_freq=3, colsample_bytree=0.5,
        reg_alpha=1.0, reg_lambda=5.0, max_depth=-1, is_unbalance=False,
        boosting_type="dart", random_state=42, verbosity=-1,
    )


def fit_and_score(features, label="run"):
    model = fit_final_model(model_factory, tuning_pool, features, label_col=LABEL)
    X_h, y_h = holdout_pool[features], holdout_pool[LABEL]
    y_pred = model.predict(X_h)
    pos_label = max(model.classes_)
    pos_col = list(model.classes_).index(pos_label)
    y_score = model.predict_proba(X_h)[:, pos_col]
    metrics = _score(tuning_pool[LABEL], y_h, y_pred, y_score, pos_label)
    return metrics, model


baseline_metrics, baseline_model = fit_and_score(SESSION_MODEL_FEATURES, "baseline")
reactive_metrics, reactive_model = fit_and_score(SESSION_MODEL_FEATURES_WITH_REACTIVE, "with reactivity")

summary = pd.DataFrame([
    {"variant": "baseline (3.7, reproduced)", "n_features": len(SESSION_MODEL_FEATURES),
     **{k: baseline_metrics[k] for k in ["accuracy", "majority_baseline_accuracy", "auc", "mcnemar_p_value"]}},
    {"variant": "+ reactive_headline/share/loudest", "n_features": len(SESSION_MODEL_FEATURES_WITH_REACTIVE),
     **{k: reactive_metrics[k] for k in ["accuracy", "majority_baseline_accuracy", "auc", "mcnemar_p_value"]}},
])
summary["edge"] = summary["accuracy"] - summary["majority_baseline_accuracy"]
print(summary.round(4).to_string(index=False))
print("\n3.7's own reported holdout number: accuracy 0.5798, edge +0.1489, auc 0.6191, p 0.0019")

                          variant  n_features  accuracy  majority_baseline_accuracy    auc  mcnemar_p_value   edge
       baseline (3.7, reproduced)          36    0.5585                      0.4309 0.5995           0.0069 0.1277
+ reactive_headline/share/loudest          39    0.5585                      0.4309 0.5649           0.0098 0.1277

3.7's own reported holdout number: accuracy 0.5798, edge +0.1489, auc 0.6191, p 0.0019


### 4. Where the new features rank, if at all

In [6]:
importance = pd.Series(
    reactive_model.feature_importances_, index=SESSION_MODEL_FEATURES_WITH_REACTIVE
).sort_values(ascending=False)
print(importance.head(15).to_string())
print()
for feat in REACTIVE_SESSION_FEATURES:
    rank = int((importance.index == feat).argmax()) + 1
    print(f"{feat}: importance {importance[feat]:.0f}, rank {rank} of {len(importance)}")

published_market-hours_share    134
fus_ceo_mean                    127
n_entity_sents                  124
momentum_5d                     124
entity_share                    111
daily_range_ratio_1d            109
momentum_1d                     109
fus_sentiment_loudest           109
reactive_share_mean             106
fus_headline_gap                 99
beta_20d                         96
relative_volume_20d              95
days_to_earnings                 93
fus_headline                     90
published_after-hours_share      89

reactive_headline_mean: importance 59, rank 28 of 39
reactive_share_mean: importance 106, rank 9 of 39
reactive_max_loudest: importance 20, rank 39 of 39


### 5. Reading it

**A caveat before the result: this notebook's baseline does not reproduce `3.7`'s
published holdout number exactly.** Accuracy 0.5585 here against `3.7`'s reported
0.5798, edge +0.1277 against +0.1489, AUC 0.5995 against 0.6191 -- close, both clear the
baseline comfortably, both land in the range `3.2`'s calibration called real, but not
identical. Same hyperparameters, same feature list, same holdout rows, same seed;
`boosting_type='dart'` and LightGBM's own version can produce slightly different trees
across environments even with `random_state` fixed, and that is the most likely
explanation, not a bug in the session-table or holdout-split code (both are asserted
against `3.7`'s exact row counts above, and pass). This does not weaken the comparison
that actually matters here, since both variants below were fit inside this same
notebook, same run, same environment -- what differs between them is only the feature
list, which is the thing being tested.

**The result: adding the three reactivity features changed nothing for the better, and
something for the worse.** Accuracy and edge are identical to four decimal places
between baseline and baseline-plus-reactivity -- not one prediction on the 188-row
holdout flipped. AUC moved from 0.5995 to 0.5649, a real drop in ranking quality even
though the hard 0.5-threshold predictions didn't change, and `mcnemar_p` moved from
0.0069 to 0.0098 -- both still comfortably significant, but the reactivity variant is
the weaker of the two on every metric that isn't identical by coincidence.

**It is not that the model ignored the new features.** `reactive_share_mean` (the
body-proxy share) ranks 9th of 39 by importance, ahead of `beta_20d`,
`relative_volume_20d`, and `days_to_earnings` -- established market features `3.6` and
`3.7` both found real signal in. The tree used it. Using it did not help, and the AUC
drop suggests it may have taken a split that would otherwise have gone to a feature
that generalises better. `reactive_headline_mean` (rank 28) and `reactive_max_loudest`
(rank 39, dead last) barely registered at all.

**This is the more direct answer to the question the lag-1 test couldn't settle.** That
test asked whether reactivity explains a correlation and came back ambiguous -- it
could not distinguish "the filter is too noisy" from "reactive articles aren't special
here." This test asks the question that actually matters for the project's goal, using
the same real headline and body-proxy scores, on the model that already proved it can
find real signal in text features when they're there (`3.7` section 4.2's importance
breakdown gave text 61.9% of the total). Here, added to that same model, reactivity
features contribute nothing positive and cost a little on ranking quality.

**Recommendation: leave it out, and don't invest in redoing the structure right now.**
Three independent checks now agree: the pattern-quality validation showed real but
limited accuracy (0.909 precision, 0.40 recall, and in-sample-tuned at that); the lag-1
split found reactive and non-reactive articles behaving identically; and this notebook
found reactivity adding nothing to the model that already works best on this corpus.
None of the three is decisive alone, but together they point the same way, and a
feature that costs AUC while contributing nothing to accuracy is not a candidate for
further investment without a reason stronger than "it might work better rebuilt." If a
future corpus or a different model surfaces a reason to revisit this, the code
(`reactive.py`, and the entity-scoped `aggregate_reactive_features()` on
`modelling/reactivity-pipeline`) is intact and tested, not deleted -- just not worth
building further on top of today.

### 6. Is this redundancy or noise?

Two readings of section 5's result are both consistent with what happened: the
reactivity features are noisy enough to add nothing, or they carry real information that
the existing market features already capture more directly, so adding them is redundant
rather than useless. These predict different correlation patterns. A noisy feature
should show weak correlation with everything, including the market state features it
would need to be redundant with. A redundant-but-real feature should correlate
meaningfully with the market features that already measure "how much is going on"
(volatility, trading range, news volume) while adding nothing on top, because that
information is already in the model through cleaner channels.

Checked directly: session-level `reactive_headline_mean`/`reactive_share_mean`/
`reactive_max_loudest` against every `MARKET_FEATURES` column, plus
`abnormal_return_1d` itself -- if reactivity tracked the actual return rather than
general market activity, that would be a different and more interesting story.

In [7]:
from scipy.stats import pearsonr

COMPARE_COLS = MARKET_SNAPSHOT + ["abnormal_return_1d"]
REACTIVE_COLS = ["reactive_headline_mean", "reactive_share_mean", "reactive_max_loudest"]

rows = []
for rc in REACTIVE_COLS:
    for cc in COMPARE_COLS:
        valid = sessions[rc].notna() & sessions[cc].notna()
        r, p = pearsonr(sessions.loc[valid, rc], sessions.loc[valid, cc])
        rows.append({"reactive_feature": rc, "compared_to": cc, "n": int(valid.sum()), "r": r, "p": p})
corr = pd.DataFrame(rows)
print(corr.pivot(index="compared_to", columns="reactive_feature", values="r").round(3).to_string())

reactive_feature      reactive_headline_mean  reactive_max_loudest  reactive_share_mean
compared_to                                                                            
abnormal_return_1d                     0.017                -0.016               -0.033
beta_20d                               0.121                 0.055                0.161
daily_range_ratio_1d                   0.198                 0.129                0.231
days_to_earnings                       0.039                -0.015               -0.012
momentum_1d                            0.021                -0.006               -0.002
momentum_20d                           0.040                -0.056                0.067
momentum_5d                           -0.001                -0.026               -0.004
news_volume                            0.186                 0.367                0.245
relative_volume_20d                    0.175                 0.116                0.156
volatility_20d                  

**A clean result.** All three reactivity features correlate meaningfully with the
same cluster of market features -- `volatility_20d` (r = 0.16-0.25), `daily_range_ratio_1d`
(r = 0.13-0.23), `news_volume` (r = 0.19-0.37), `relative_volume_20d` (r = 0.16-0.18),
all p < 0.0001 -- and `reactive_max_loudest` against `news_volume` is the single
strongest correlation in the table at r = 0.367. That is exactly the fingerprint of
"how much is going on right now," the same quantity `volatility_20d`/
`daily_range_ratio_1d`/`news_volume` already measure directly and continuously, rather
than through a count of sentences matching a price-move regex.

**And critically, correlation with the actual return is close to zero.** Against
`abnormal_return_1d` itself, all three reactivity features sit at r = 0.02 to -0.03 --
no relationship worth naming. Reactivity tracks *activity level*, not *direction*, and
the model's actual job is direction. That is the mechanism section 5's result needed: a
feature can be real (correlated with genuine market state, which is why the tree used
`reactive_share_mean`) and still contribute nothing to accuracy, because the state it
tracks is already available through cleaner, more precise numeric channels, and it was
never correlated with the thing being predicted in the first place.

**This favours redundancy over noise as the explanation, though it does not fully rule
noise out.** A genuinely noisy feature would not be expected to correlate this
consistently (four separate market features, all p < 0.0001) with a real, independently
measured quantity. The more likely story: `reactive_share_mean` earned its rank 9
because it partially substitutes for `news_volume`/`volatility_20d` when the tree needs
that information and one of the cleaner columns isn't available at a given split, not
because it adds anything those columns don't already have. That would also explain the
AUC cost -- a redundant, noisier proxy competing with a cleaner signal for the same
splits can make the ensemble's probability calibration slightly worse even while leaving
hard 0.5-threshold predictions unchanged.

**Revised recommendation, same conclusion.** This does not change section 5's "leave it
out" -- if anything it sharpens the reason why. The gap is not "the filter is too
inaccurate to find anything real," which a corpus-scale, entity-scoped rebuild could
fix. It is "what it finds is already measured better elsewhere," which a more accurate
reactivity filter would not fix, because the problem is not measurement error, it is
that reactivity-as-activity-level and reactivity-as-direction-signal are different
things, and only the first one is what a sentence-counting regex was ever going to
find.